# Food Price Watch 02: model selection and tuning
Code comes from GitHub (`main`) into temporary Colab storage each session; Google Drive (`MyDrive/FoodPriceWatch`) holds data and outputs only. S24 one-model design check, S25 six-candidate validation table (2024), S26 Optuna tuning and the pre-registration file. Not implemented yet. Run one stage at a time; never use Run all.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Fresh clone of main (or fast-forward if this runtime already has it).
import os
if not os.path.isdir('/content/foodprice_watch/.git'):
    !git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else:
    !git -C /content/foodprice_watch pull --ff-only origin main
%pip install -q -r /content/foodprice_watch/requirements-colab.txt

In [ ]:
%cd /content/foodprice_watch
%env PYTHONPATH=/content/foodprice_watch/src
%env FPW_ARTIFACT_ROOT=/content/drive/MyDrive/FoodPriceWatch
from pathlib import Path
import csv, hashlib, json, shutil
ROOT = Path('/content/drive/MyDrive/FoodPriceWatch')
for folder in ['inbox/project', 'inbox/external', 'data/processed', 'outputs', 'reports/tables',
               'reports/figures', 'reports/project/tables', 'state', 'config', 'logs', 'models']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
# First run only: copy the starting state files into Drive. Existing files and seals are never replaced.
for name in ['state/stages.json', 'config/item_driver_map.yaml', 'DECISIONS.md', 'PROGRESS.md', 'DEVIATIONS.md',
             'reports/project/tables/P2_audit.csv', 'reports/project/tables/P2_audit_summary.csv']:
    if not (ROOT / name).exists():
        shutil.copy2(name, ROOT / name)
# Remove half-written files left by an interrupted session.
for folder in ['outputs', 'data']:
    for tmp in (ROOT / folder).rglob('*.tmp'):
        tmp.unlink()
CODE_COMMIT = !git rev-parse HEAD
CODE_COMMIT = CODE_COMMIT[0]
CHECKS_PASS = False
stage_ok = {}

In [ ]:
CHECKS_PASS = False
stage_ok.clear()
!python -m fpw.check
CHECKS_PASS = get_ipython().user_ns['_exit_code'] == 0
assert CHECKS_PASS, 'Startup check failed; fix it before running stages'

## Run the stages of this notebook
Set `STAGES` if needed and run the stage cell, then the summary cell. A stage that is already done is skipped; the first failure stops the notebook. Paste the printed summary back.

In [ ]:
STAGES = ['S24', 'S25', 'S26']   # run in order; a stage that is already done is skipped
assert CHECKS_PASS, 'Run and pass startup checks first'
stage_ok.clear()
for STAGE in STAGES:
    stage_ok[STAGE] = False
    !python -m fpw.runner {STAGE}
    stage_ok[STAGE] = get_ipython().user_ns['_exit_code'] == 0
    assert stage_ok[STAGE], f'{STAGE} failed; stop and review output'

In [ ]:
def summarize_unit(sid):
    assert stage_ok.get(sid), 'Stage command did not succeed in this session'
    ledger = json.loads((ROOT / 'state/stages.json').read_text())
    stage = next(s for s in ledger['stages'] if s['id'] == sid)
    assert stage['status'] == 'done', 'STOP: stage was not finalized'
    summary = []
    for name, expected in stage['output_sha256'].items():
        path = ROOT / name
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        assert actual == expected, 'STOP: output hash mismatch: ' + name
        rows = None
        if path.suffix == '.csv':
            with path.open(newline='') as f:
                rows = sum(1 for _ in csv.DictReader(f))
        elif path.suffix == '.parquet':
            import pyarrow.parquet as pq
            rows = pq.read_metadata(path).num_rows
        summary.append(dict(file=name, rows=rows, sha256=actual))
    assert summary, 'STOP: no finalized output hashes'
    print(json.dumps(dict(unit=sid, outputs=summary, check='PASS', code_commit=CODE_COMMIT), indent=2))

for STAGE in STAGES:
    summarize_unit(STAGE)